In [71]:
!pip install kaggle

**Importing the Dependencies**

In [72]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

**Data Collection- Kaggle API**

In [73]:
kaggle_dictionary = json.load(open("kaggle.json"))

In [74]:
kaggle_dictionary.keys()

dict_keys(['username', 'key'])

In [75]:
# setup kaggle credentials as environment variables
os.environ["KAGGLE_USERNAME"] = kaggle_dictionary["username"]
os.environ["KAGGLE_KEY"] = kaggle_dictionary["key"]

In [76]:
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [77]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


In [78]:
# unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
  zip_ref.extractall()

In [79]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


**Loading teh Dataset**

In [80]:
data = pd.read_csv("/content/IMDB Dataset.csv")

In [81]:
data.shape

(50000, 2)

In [82]:
data.sample(5,  random_state=1)

,review,sentiment
26247,With No Dead Heroes you get stupid lines like ...,negative
35067,I thought maybe... maybe this could be good. A...,negative
34590,An elite American military team which of cours...,negative
16668,Ridiculous horror film about a wealthy man (Jo...,negative
12196,"Well, if you are one of those Katana's film-nu...",positive


## Text preprocessing
>  convert text to lower case

>  remove html tags and urls

> remove punctuaions

> chatword treatment

> stop words remove


In [84]:
#  to lower case
data['review'] =data['review'].str.lower()

In [85]:
# remove html tags
from bs4 import BeautifulSoup

def remove_html_tags(text):
    return BeautifulSoup(text, "html.parser").get_text()

# Apply on clean_overview
data['review'] = data['review'].apply(remove_html_tags)

In [86]:
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [87]:
import re

# 1) Compile the regex once (handles http(s) and www links)
url_pattern = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)

# 2) A small helper that uses the compiled pattern
def remove_urls(text):
    if not isinstance(text, str):
        return ''            # handle NaNs or non-strings safely
    return url_pattern.sub('', text).strip()

# 3) Apply to the column (row by row)
data['review'] = data['review'].apply(remove_urls)

data.head()


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


In [88]:
import re
import string

# 1) Compile a regex for punctuation
punct_pattern = re.compile('[%s]' % re.escape(string.punctuation))


def remove_punctuation(text):
    if not isinstance(text, str):
        return ''
    return punct_pattern.sub('', text)

# 3) Apply to the column
data['review'] = data['review'].apply(remove_punctuation)

data.head()


,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [89]:
# 1) Create a small dictionary for common chat words
chat_words = {
    "u": "you",
    "r": "are",
    "ur": "your",
    "btw": "by the way",
    "idk": "i do not know",
    "lol": "laughing out loud",
    "thx": "thanks",
    "pls": "please",
    "plz": "please",
    "omg": "oh my god",
    "gonna": "going to",
    "wanna": "want to",
    "im": "i am",
    "dont": "do not",
    "cant": "cannot",
    "doesnt": "does not",
    "isnt": "is not",
    "wasnt": "was not",
    "shouldnt": "should not",
    "couldnt": "could not",
    "wouldnt": "would not",
    "ive": "i have",
    "id": "i would",
    "didnt": "did not",
    "hru": "how are you",
    "brb": "be right back",
    "ttyl": "talk to you later",
    "gr8": "great",
    "b4": "before",
    "imo": "in my opinion",
    "fyi": "for your information",
    "smh": "shaking my head",
    "lmk": "let me know",
    "np": "no problem",
    "ty": "thank you",
    "yw": "you are welcome"
}

# 2) Define a function to expand them
def expand_chat_words(text):
    if not isinstance(text, str):
        return ''
    words = text.split()  # split into list of words
    expanded = [chat_words.get(w, w) for w in words]  # replace if found
    return ' '.join(expanded)

# 3) Apply on the overview column
data['review'] = data['review'].apply(expand_chat_words)




In [90]:
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [91]:
!pip install pyspellchecker


In [93]:
#  remove stop words
import nltk
from nltk.corpus import stopwords

# Download stopwords (only once)
nltk.download('stopwords')

# Get English stopword list
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    if not isinstance(text, str):
        return ''
    words = text.split()
    filtered = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered)

# Apply on the overview column
data['review'] = data['review'].apply(remove_stopwords)

data.sample(5, random_state =1)



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,review,sentiment
26247,dead heroes get stupid lines like woefully aby...,negative
35067,thought maybe maybe could good early appearanc...,negative
34590,elite american military team course happens in...,negative
16668,ridiculous horror film wealthy man john carrad...,negative
12196,well one katanas filmnuts like sure appreciate...,positive


In [94]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [95]:
# change the ouput labels to numeric (binary in this case)
def label_to_binary(label):
    if label == "positive":
        return 1
    else:
        return 0

data['sentiment'] = data['sentiment'].apply(label_to_binary)


In [96]:
data.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically theres family little boy jake thinks...,0
4,petter matteis love time money visually stunni...,1


In [97]:
data["sentiment"].value_counts()

,count
sentiment,
1,25000
0,25000


In [98]:
# split data into training data and test data
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [99]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


In [100]:
train_data['sentiment'].value_counts()

,count
sentiment,
0,20039
1,19961


**Data Preprocessing**

In [101]:
MAX_WORDS = 5000  # top words to keep
MAX_LEN = 200     # max length of sequences

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(train_data["review"])  # build vocab from training data

# Convert texts to sequences of integers
X_train_seq = tokenizer.texts_to_sequences(train_data["review"])
X_test_seq = tokenizer.texts_to_sequences(test_data["review"])

# Pad sequences so each has length MAX_LEN
X_train = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test = pad_sequences(X_test_seq, maxlen=MAX_LEN)


In [102]:
print(X_train)

[[   0    0    0 ...  102  224 3629]
 [   0    0    0 ...   14 1599   28]
 [   0    0    0 ...  195  599   11]
 ...
 [   0    0    0 ... 1000 1472  507]
 [   0    0    0 ... 1138  137   28]
 [   0    0    0 ...  360   12 1807]]


In [103]:
print(X_test)

[[   0    0    0 ...   95  844 2810]
 [   0    0    0 ...   28 1743   10]
 [   0    0    0 ...  821  929   21]
 ...
 [   0    0    0 ...  465   87 3181]
 [   0    0    0 ...  110   65 2075]
 [   0    0    0 ...   10  223    3]]


In [104]:
Y_train = train_data["sentiment"]
Y_test = test_data["sentiment"]

In [105]:
print(Y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


In [106]:
!pip install gensim

In [107]:
import gensim.downloader as api
import numpy as np

# Load GloVe embeddings (100d)
glove = api.load("glove-wiki-gigaword-100")
embedding_dim = 100


In [108]:
#  Build Embedding Matrix

vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if i >= MAX_WORDS:
        continue
    if word in glove:
        embedding_matrix[i] = glove[word]
    else:
        # Random vector for out-of-vocabulary words
        embedding_matrix[i] = np.random.normal(0, 0.01, embedding_dim)


In [114]:

embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    input_length=MAX_LEN,
    trainable=True  # unfreeze the vectors
)

**LSTM - Long Short-Term Memory**

In [119]:
# build the model
from tensorflow.keras.layers import Bidirectional, LSTM

model = Sequential()
model.add(embedding_layer) # input_dim -size of your vocabulary. # ouput_dim: size of the embedding vector # input_length : size of each input sequence

model.add(Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2)))
model.add(Dense(1, activation="sigmoid"))

In [120]:
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (64, 200, 100)         │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,000 (1.91 MB)

 Trainable params: 500,000 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [121]:
# compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

**Training the Model**

In [122]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 641s 1s/step - accuracy: 0.7071 - loss: 0.5500 - val_accuracy: 0.8631 - val_loss: 0.3235
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 634s 1s/step - accuracy: 0.8656 - loss: 0.3180 - val_accuracy: 0.8890 - val_loss: 0.2655
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 622s 1s/step - accuracy: 0.8980 - loss: 0.2559 - val_accuracy: 0.8931 - val_loss: 0.2585
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 634s 1s/step - accuracy: 0.9072 - loss: 0.2295 - val_accuracy: 0.8899 - val_loss: 0.2637
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 621s 1s/step - accuracy: 0.9215 - loss: 0.2033 - val_accuracy: 0.8924 - val_loss: 0.2617


**Model Evaluation**

In [123]:
loss, accuracy = model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 63s 173ms/step - accuracy: 0.8940 - loss: 0.2612
Test Loss: 0.26387324929237366
Test Accuracy: 0.8938999772071838


**Building a Predictive System**

In [133]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''

    # lowercase
    text = text.lower()

    # remove html tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # remove urls
    text = url_pattern.sub('', text).strip()

    # remove punctuation
    text = punct_pattern.sub('', text)

    # expand chat words
    words = text.split()
    words = [chat_words.get(w, w) for w in words]

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    return ' '.join(words)


In [134]:
def predict_sentiment(review):
    # preprocess
    review = preprocess_text(review)

    # tokenize and pad
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_LEN)

    # predict
    prediction = model.predict(padded_sequence)
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment


In [141]:
# example usage
new_review = "It wasn’t terrible, but it certainly wasn’t good either."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step
The sentiment of the review is: negative


In [136]:
# example usage
new_review = "I couldn’t connect with any of the characters. Very disappointing."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
The sentiment of the review is: negative


In [137]:
# example usage
new_review = "This movie was ok but not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step
The sentiment of the review is: positive


In [138]:
new_review  ="I wanted to love this film because of the hype, but it just didn’t live up to my expectations."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
The sentiment of the review is: negative


In [139]:
!pip install tensorflow keras pandas numpy scikit-learn streamlit matplotlib


In [140]:
# Save model
model.save("sentiment_lstm_model.h5")

# Save tokenizer
import pickle
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
